# 01_dataclean_1_da_prices.ipynb

Refactored short version using shared utility functions.


In [7]:
import pandas as pd
from cleaning_utils import run_source_pipeline, add_calendar_columns, aggregate_hourly
from verify_time_consistency import run_checks, run_hourly_checks


In [8]:
files = [
    "../../data/da_prices/original/GUI_ENERGY_PRICES_201412312300-201512312300.csv",
    "../../data/da_prices/original/GUI_ENERGY_PRICES_201512312300-201612312300.csv",
    "../../data/da_prices/original/GUI_ENERGY_PRICES_201612312300-201712312300.csv",
    "../../data/da_prices/original/GUI_ENERGY_PRICES_201712312300-201812312300 - 1.csv",
    "../../data/da_prices/original/GUI_ENERGY_PRICES_201712312300-201812312300 - 2.csv",
    "../../data/da_prices/original/GUI_ENERGY_PRICES_201812312300-201912312300.csv",
    "../../data/da_prices/original/GUI_ENERGY_PRICES_201912312300-202012312300.csv",
    "../../data/da_prices/original/GUI_ENERGY_PRICES_202012312300-202112312300.csv",
    "../../data/da_prices/original/GUI_ENERGY_PRICES_202112312300-202212312300.csv",
    "../../data/da_prices/original/GUI_ENERGY_PRICES_202212312300-202312312300.csv",
    "../../data/da_prices/original/GUI_ENERGY_PRICES_202312312300-202412312300.csv",
    "../../data/da_prices/original/GUI_ENERGY_PRICES_202412312300-202512312300.csv",
]

KEEP_COLS = ['Sequence', 'MTU (CET/CEST)', 'Day-ahead Price (EUR/MWh)']
RENAME_MAP = {'MTU (CET/CEST)': 'period', 'Day-ahead Price (EUR/MWh)': 'price'}
VALUE_COLS = ['price']
OUTPUT_CSV = '../../data_cleaned/by_source/01_ENERGY_PRICES.csv'
ROW_SELECTION_COL = 'Sequence'
ROW_SELECTION_DROP_VALUE = 'Sequence Sequence 2'


In [9]:
result = run_source_pipeline(
    files=files,
    keep_cols=KEEP_COLS,
    rename_map=RENAME_MAP,
    value_cols=VALUE_COLS,
    row_selection_col=ROW_SELECTION_COL,
    row_selection_drop_value=ROW_SELECTION_DROP_VALUE,
    include_calendar_columns=False,
)

raw_df = result.raw
df_utc_q = result.quarter_hour

# Calendar features are intentionally added here (deferred from source cleaning helper)
df_utc_q_cal = add_calendar_columns(df_utc_q.copy())
df_utc_h = aggregate_hourly(df_utc_q_cal, value_cols=VALUE_COLS)

raw_df.shape, df_utc_q.shape, df_utc_h.shape


((541275, 6), (287067, 4), (96336, 12))

In [10]:
# Consistency checks moved to separate script/module
# Quarter-hour consistency before calendar feature engineering
run_checks(df_utc_q, ts_col='period_start_utc')

# Hourly completeness after calendar engineering + mean aggregation
run_hourly_checks(df_utc_h, ts_col='period_start_utc')


--- Frequency summary ---
period_start_utc
0 days 00:15:00    254307
0 days 01:00:00     32759
Name: count, dtype: int64

--- 15-min completeness (expected 96/day) ---
is_complete
True     2648
False    1367
Name: count, dtype: int64
Sample incomplete 15-min days:
                  n_periods  expected  is_complete
period_start_utc                                  
2015-01-04                1        96        False
2015-01-05               24        96        False
2015-01-06               24        96        False
2015-01-07               24        96        False
2015-01-08               24        96        False

--- Hourly completeness (expected 24/day) ---
is_complete
True     4013
False       2
Name: count, dtype: int64
Sample incomplete hourly days:
                  n_periods  expected  is_complete
period_start_utc                                  
2015-01-04                1        24        False
2025-12-31               23        24        False


In [11]:
df_utc_h.head()


,date,year,month,day,dayofyear,hour,week,dayofweek,price,period_start_utc,period_end_utc,c_by_hour
0,2015-01-04,2015,1,4,4,23,1,6,22.34,2015-01-04 23:00:00,2015-01-05 00:00:00,1
1,2015-01-05,2015,1,5,5,0,2,0,17.93,2015-01-05 00:00:00,2015-01-05 01:00:00,1
2,2015-01-05,2015,1,5,5,1,2,0,15.17,2015-01-05 01:00:00,2015-01-05 02:00:00,1
3,2015-01-05,2015,1,5,5,2,2,0,16.38,2015-01-05 02:00:00,2015-01-05 03:00:00,1
4,2015-01-05,2015,1,5,5,3,2,0,17.38,2015-01-05 03:00:00,2015-01-05 04:00:00,1


In [12]:
df_utc_h.to_csv(OUTPUT_CSV, index=False)
print('saved:', OUTPUT_CSV)


saved: ../../data_cleaned/by_source/01_ENERGY_PRICES.csv
